<a href="https://colab.research.google.com/github/dokhoivi0912/ISYS2001---Khoi-Vi-Do/blob/main/Assessment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assessment 2: Student Budget Coach Application

## Step 1: Understand the Problem
College students often struggle to manage their monthly income and allocate funds effectively across necessary living expenses, personal wants, and savings goals.
This application provides students with a personal finance tool. By taking the user's monthly income or processing an uploaded CSV file of transaction records, the application automatically calculates budget allocations based on the standard 50/30/20 financial rule (50% Essentials, 30% Personal Wants, 20% Savings). Furthermore, it integrates a Gemini AI assistant to deliver personalized financial guidance grounded in the student's actual budget data.

## Step 2: Identify Inputs and Outputs
* **Inputs:**
  - `income` (float): The user's monthly income (must be a positive numerical value).
  - `transactions_csv` (file / string path, optional): An uploaded CSV file containing transaction records (Date, Category, Amount, Description) processed using Pandas for data grounding.
  - `user_question` (string): Specific questions or financial advice requests provided by the user to the AI advisor.
* **Outputs:**
  - `budget_summary` (string/dictionary): A structured financial breakdown displaying the exact calculated dollar amounts for Essentials, Wants, and Savings.
  - `ai_advice` (string): Tailored financial insights and suggestions generated by the Gemini AI assistant grounded in the calculated budget numbers and transaction records.

## Step 3: Worked Example by Hand
Suppose a student enters a monthly income of **$1,000.00** (or has a processed total income of $1,000.00 from their CSV file):
- **Essentials (50%):** $1,000.00 * 0.50 = $500.00
- **Personal Wants (30%):** $1,000.00 * 0.30 = $300.00
- **Savings (20%):** $1,000.00 * 0.20 = $200.00

*Edge Case & Invalid Input Handling:*
If a user inputs an invalid income such as **-$100.00** or **$0.00**:
- **Expected Output:** "Error: Income must be a positive number greater than zero."

## Step 4: Write Pseudocode
```text
FUNCTION calculate_budget(income):
    IF income <= 0 THEN
        RETURN "Error: Income must be a positive number greater than zero."
    ENDIF
    
    SET essentials = income * 0.50
    SET wants = income * 0.30
    SET savings = income * 0.20
    
    RETURN "Essentials (50%): $" + essentials + " | Wants (30%): $" + wants + " \vert{} Savings (20\%): $" + savings
END FUNCTION

In [ ]:
# ==========================================
# Requirement 3: Custom Tool Function (R3)
# ==========================================

def calculate_budget(income):
    """
    Calculates 50/30/20 budget allocations for a given monthly income.
    Inputs: income (float or int)
    Outputs: Formatted string summary or error message for edge cases.
    """
    # Edge case check for invalid input (zero or negative income)
    if income <= 0:
        return "Error: Income must be a positive number greater than zero."

    # Calculate budget breakdown based on 50/30/20 rule
    essentials = income * 0.50
    wants = income * 0.30
    savings = income * 0.20

    return f"Essentials (50%): ${essentials:.2f} | Wants (30%): ${wants:.2f} | Savings (20%): ${savings:.2f}"

# ==========================================
# Requirement 5: Assert-based Testing (R5)
# ==========================================

# Test Case 1: Standard input ($1000)
assert calculate_budget(1000) == "Essentials (50%): $500.00 | Wants (30%): $300.00 | Savings (20%): $200.00"

# Test Case 2: Boundary/Edge Case input ($0)
assert calculate_budget(0) == "Error: Income must be a positive number greater than zero."

# Test Case 3: Invalid input (Negative value -$50)
assert calculate_budget(-50) == "Error: Income must be a positive number greater than zero."

# Test Case 4: Decimal/Float input ($1500.50) -> Fixed $301.10 to $300.10
assert calculate_budget(1500.50) == "Essentials (50%): $750.25 | Wants (30%): $450.15 | Savings (20%): $300.10"

print("✅ All test cases passed successfully!")

✅ All test cases passed successfully!


In [ ]:
# ==========================================
# Requirement 1, 2 & Module 7: Pandas CSV & Gemini API
# ==========================================

import pandas as pd
import google.generativeai as genai
from google.colab import userdata

# 1. Custom Tool Function from Phase 2 (R3)
def calculate_budget(income):
    """
    Calculates 50/30/20 budget allocations for a given monthly income.
    """
    if income <= 0:
        return "Error: Income must be a positive number greater than zero."

    essentials = income * 0.50
    wants = income * 0.30
    savings = income * 0.20

    return f"Essentials (50%): ${essentials:.2f} | Wants (30%): ${wants:.2f} | Savings (20%): ${savings:.2f}"

# 2. Load and process CSV data via Pandas (Requirement R2 / Module 7)
def process_csv_income(csv_path="student_transactions.csv"):
    """
    Reads student_transactions.csv and calculates the total net income from transactions.
    """
    try:
        df = pd.read_csv(csv_path)
        net_income = float(df["Amount"].sum())
        return net_income
    except Exception as e:
        print(f"Error processing CSV file: {e}")
        return None

# 3. Setup Gemini API Credentials (Requirement R1 / Module 8)
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=API_KEY)
except Exception as e:
    print("⚠️ API Key Warning: Please set 'GEMINI_API_KEY' in Colab Secrets tab.")

# 4. System Instruction for Student Finance Advisor Persona (R1)
system_instruction = (
    "You are an empathetic, professional personal finance advisor for university students. "
    "Analyze the student's 50/30/20 budget breakdown (Essentials 50%, Wants 30%, Savings 20%) "
    "and answer their questions based strictly on their actual financial figures. "
    "Keep responses practical, encouraging, and concise (under 3-4 sentences)."
)

model = genai.GenerativeModel(
    model_name="gemini-flash-latest",
    system_instruction=system_instruction
)

# 5. Grounded AI Advice Function (R1 & R2)
def ask_gemini_advisor(income, user_question):
    """
    Connects Custom Tool data (R3) with Gemini AI (R1) grounded in user input / CSV data (R2).
    """
    budget_info = calculate_budget(income)

    if "Error:" in budget_info:
        return budget_info

    prompt = (
        f"User Monthly Budget Summary: {budget_info}\n"
        f"User Question: {user_question}"
    )

    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error communicating with Gemini AI: {str(e)}"

# Test Case: Grounding AI advice with processed CSV total ($60.55)
sample_csv_income = process_csv_income("student_transactions.csv")
if sample_csv_income is not None:
    print(f"✅ Processed Net Income from CSV: ${sample_csv_income:.2f}")
    sample_question = "How much can I spend on wants based on this budget?"
    print("--- Testing Gemini Advisor Integration ---")
    print(ask_gemini_advisor(sample_csv_income, sample_question))

✅ Processed Net Income from CSV: $60.55
--- Testing Gemini Advisor Integration ---


ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2478.11ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2004.77ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2932.41ms


Based on your budget, you have **$18.16** allocated to spend on your "wants" each month. While that might feel a bit tight on a student budget, being intentional with it can still leave room for a small treat or fun outing. You're doing a fantastic job establishing healthy financial habits by tracking your spending!


In [ ]:
# ==========================================
# Requirement 4: Gradio Web Interface (R4)
# Fully Consolidated App with All Dependencies
# ==========================================

import gradio as gr
import pandas as pd
import google.generativeai as genai
from google.colab import userdata

# ------------------------------------------
# 1. Custom Tool Function (R3)
# ------------------------------------------
def calculate_budget(income):
    try:
        income = float(income)
    except (ValueError, TypeError):
        return "Error: Invalid input. Income must be a numeric value."

    if income <= 0:
        return "Error: Income must be a positive number greater than zero."

    essentials = income * 0.50
    wants = income * 0.30
    savings = income * 0.20

    return f"Essentials (50%): ${essentials:.2f} | Wants (30%): ${wants:.2f} | Savings (20%): ${savings:.2f}"

# ------------------------------------------
# 2. Pandas CSV Processor Function (R2 / Module 7)
# ------------------------------------------
def process_csv_income(csv_file_path):
    try:
        df = pd.read_csv(csv_file_path)
        net_income = float(df["Amount"].sum())
        return net_income
    except Exception as e:
        print(f"Error processing CSV: {e}")
        return None

# ------------------------------------------
# 3. Gemini API Integration Setup (R1 / Module 8)
# ------------------------------------------
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=API_KEY)
except Exception:
    pass

system_instruction = (
    "You are an empathetic, professional personal finance advisor for university students. "
    "Analyze the student's 50/30/20 budget breakdown and answer their questions based strictly "
    "on their actual financial figures. Keep responses practical and concise (under 3-4 sentences)."
)

model = genai.GenerativeModel(
    model_name="gemini-flash-latest",
    system_instruction=system_instruction
)

def ask_gemini_advisor(income, user_question):
    budget_info = calculate_budget(income)
    if "Error:" in budget_info:
        return budget_info

    prompt = f"User Monthly Budget Summary: {budget_info}\nUser Question: {user_question}"
    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error communicating with Gemini AI: {str(e)}"

# ------------------------------------------
# 4. Gradio Wrapper Functions (R4)
# ------------------------------------------
def run_budget_calculator(income_input):
    return calculate_budget(income_input)

def run_csv_processor(file_obj):
    if file_obj is None:
        return "Error: Please upload a valid CSV file."

    net_income = process_csv_income(file_obj.name)
    if net_income is None or net_income <= 0:
        return "Error: Could not calculate valid income from the uploaded CSV."

    budget_summary = calculate_budget(net_income)
    return f"Processed CSV Net Income: ${net_income:.2f}\n\n{budget_summary}"

def run_ai_advisor(income_input, user_question):
    if not user_question.strip():
        return "Please enter a question for the AI Advisor."
    return ask_gemini_advisor(income_input, user_question)

# ------------------------------------------
# 5. Build & Launch Gradio Blocks Interface
# ------------------------------------------
with gr.Blocks(title="Student Budget Coach") as demo:
    gr.Markdown("# 🎓 Student Budget Coach & Personal Finance Assistant")
    gr.Markdown(
        "Manage your monthly student budget using the standard 50/30/20 financial rule. "
        "Enter your income manually or upload transaction CSV files, then consult your AI Advisor!"
    )

    with gr.Tab("1. Manual Budget Calculator"):
        gr.Markdown("### Calculate 50/30/20 Allocations")
        income_in = gr.Number(label="Monthly Income ($)", value=1000.0)
        calc_btn = gr.Button("Calculate Budget")
        budget_out = gr.Textbox(label="Budget Breakdown Summary", interactive=False)
        calc_btn.click(fn=run_budget_calculator, inputs=income_in, outputs=budget_out)

    with gr.Tab("2. Process CSV Transactions"):
        gr.Markdown("### Upload Transaction File (Pandas Integration)")
        csv_in = gr.File(label="Upload CSV File (e.g., student_transactions.csv)", file_types=[".csv"])
        csv_btn = gr.Button("Process CSV File")
        csv_out = gr.Textbox(label="CSV Analysis & Budget Breakdown", interactive=False)
        csv_btn.click(fn=run_csv_processor, inputs=csv_in, outputs=csv_out)

    with gr.Tab("3. Gemini AI Financial Advisor"):
        gr.Markdown("### Consult Your AI Finance Coach")
        chat_income = gr.Number(label="Monthly Income Base ($)", value=1000.0)
        chat_question = gr.Textbox(
            label="Your Question",
            placeholder="e.g., Can I afford to buy a $150 concert ticket this month?"
        )
        chat_btn = gr.Button("Ask Gemini Advisor")
        chat_out = gr.Textbox(label="AI Advisor Recommendation", lines=4, interactive=False)
        chat_btn.click(fn=run_ai_advisor, inputs=[chat_income, chat_question], outputs=chat_out)

demo.launch(share=True, debug=True)

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9d59e7225bd5e17769.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
